# Chapter 2: Build GPT From Scratch

Character-level GPT trained on tiny-Shakespeare. Every stage builds on the last — run cells in order.

See `../RUNBOOK.md` for the why/enterprise-notes/interview-angles, and `WALKTHROUGH.md` in this folder for the deep-dive line-by-line breakdown. Code cells below are heavily commented so the notebook itself is a usable reference too, not just a script to run.

## Stage 1: Data loading + character-level tokenization

In [ ]:
# Download the tiny Shakespeare dataset (~1MB, ~1.1M characters).
# -nc = "no clobber": skip re-downloading if input.txt already exists in this directory.
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
# Read the entire file into memory as one big Python string.
# At this scale (~1MB) that's totally fine; real corpora (Chapter 4) get streamed/chunked instead.
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")
print("\nFirst 300 characters:\n")
print(text[:300])

### Build the vocabulary
Find every unique character in the dataset — that's our entire vocabulary. No subwords, no BPE merges yet (that's Chapter 4).

In [ ]:
# set(text) collapses the string down to its unique characters (order lost);
# sorted(...) gives us a deterministic, reproducible ordering -> the same character
# always gets the same integer ID across runs, which matters for reproducibility.
chars = sorted(list(set(text)))

# vocab_size is the single most important number in this notebook -- it determines
# the width of every embedding table and every output layer we build from here on.
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"All characters: {''.join(chars)!r}")  # includes letters, punctuation, newline, space, etc.

### Encoder / decoder
`stoi` (string-to-int) and `itos` (int-to-string) are the entire tokenizer. `encode` and `decode` are just dictionary lookups.

In [ ]:
# Two dictionaries are our ENTIRE tokenizer -- this is the whole "tokenization" step
# for a character-level model. Compare this to Chapter 4, where we'll train a real
# BPE tokenizer with thousands of learned subword merges instead of ~65 raw characters.
stoi = {ch: i for i, ch in enumerate(chars)}   # e.g. {'a': 40, 'b': 41, ...}
itos = {i: ch for i, ch in enumerate(chars)}   # e.g. {40: 'a', 41: 'b', ...}

# encode: string -> list[int]  (look up each character's integer ID)
encode = lambda s: [stoi[c] for c in s]
# decode: list[int] -> string  (look up each ID's character, then join them back together)
decode = lambda l: ''.join([itos[i] for i in l])

# Round-trip sanity check: encoding then decoding should return the exact original string.
test = "Hello"
encoded = encode(test)
print(f"{test!r} encoded: {encoded}")
print(f"Decoded back: {decode(encoded)!r}")

### Encode the entire dataset into a tensor

In [ ]:
import torch

# Encode the WHOLE dataset in one shot: every character in the 1.1M-character
# corpus becomes one integer. dtype=torch.long because these are indices into an
# embedding table -- PyTorch's embedding/index ops require integer (long) tensors.
data = torch.tensor(encode(text), dtype=torch.long)

print(f"Data shape: {data.shape}, dtype: {data.dtype}")
print(f"First 100 encoded characters:\n{data[:100]}")

### Train / validation split (90 / 10)
Never evaluate on data the model trained on — validation loss is our only honest signal for overfitting.

In [ ]:
# Simple contiguous split (first 90% / last 10%) rather than a random shuffle --
# this is a text-continuation task, so we want validation data the model has
# genuinely never seen any part of, not just individually-shuffled characters
# from the same passages it trained on.
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training characters:   {len(train_data):,}")
print(f"Validation characters: {len(val_data):,}")

### Context windows (`block_size`)
The model never sees the whole dataset at once — only a fixed-size window. One window of length `block_size` actually yields `block_size` separate training examples (predict the next char at every position).

In [ ]:
# block_size = the context window / how many characters of history the model can see
# at once. Small on purpose here (8) so every individual training example below can
# be printed and inspected by eye. GPT-2 uses 1024; modern LLMs use tens of thousands.
block_size = 8

# x = the input characters, y = x shifted one position to the right (the "next char"
# targets). This single offset trick is how autoregressive next-token training works.
x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("One block of text yields these context -> target pairs:\n")
for t in range(block_size):
    context = x[:t+1]        # everything seen so far, growing one character at a time
    target = y[t]             # the single character that should come next
    print(f"context: {decode(context.tolist())!r:20} -> target: {decode([target.item()])!r}")

### Batching
Stack many independent context windows together so the GPU processes them in parallel instead of one at a time.

In [ ]:
# Fix the random seed so results are reproducible across runs/machines -- useful when
# comparing loss curves later, since "did my change help" is only meaningful if the
# random batches being compared are otherwise identical.
torch.manual_seed(1337)

batch_size = 4  # how many independent sequences we process simultaneously (this is "B")

def get_batch(split):
    """Sample a random batch of (input, target) sequence pairs from train or val data."""
    data_split = train_data if split == 'train' else val_data

    # Pick batch_size random starting positions. Upper bound is
    # len(data_split) - block_size so every window of length block_size stays in-bounds.
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # For each starting position i, grab a block_size-length window as input,
    # and the SAME window shifted one character later as the target -- exactly
    # the same offset trick as the single-example version above, just batched.
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)   # (batch_size, block_size) = (4, 8)
print(xb)
print('targets:', yb.shape)  # same shape -- one target character per input position
print(yb)

**Checkpoint:** at this point you have `get_batch('train')` / `get_batch('val')` producing `(batch_size, block_size)` integer tensors ready to feed into embeddings. Next: the Bigram Language Model — the simplest possible baseline, built before self-attention so we can *feel* why it fails.

## Stage 2: The Bigram Language Model (baseline)

See `WALKTHROUGH.md` in this folder for the full line-by-line breakdown, shape tracing, and the guide-dog/blind-hiker training analogy.

This is deliberately the *dumbest possible* language model: given one character, predict the next, with zero memory of anything before it. We build and train it first so the self-attention section (next) has something concrete to improve on.

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # This single embedding table IS the entire model. Row i holds vocab_size
        # raw scores ("logits") for what character comes after character i.
        # There's no hidden layer, no mixing of information across positions --
        # each character's prediction depends ONLY on that one character.
        # Shape: (vocab_size, vocab_size), e.g. 65x65 for our Shakespeare vocab.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) integer tensor of character indices.
        # Looking up idx in the embedding table gives, for every position, that
        # character's row of "next character" scores.
        logits = self.token_embedding_table(idx)  # (B, T, C) where C = vocab_size

        if targets is None:
            # Inference/generation mode: no ground truth to compare against, so skip loss.
            loss = None
        else:
            # F.cross_entropy expects predictions as (N, C) and targets as (N,) --
            # i.e. a flat list of examples, not a (B, T, C) 3D tensor. Flattening
            # collapses batch and time together: every (sequence, position) pair
            # becomes one independent training example.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """Autoregressively sample max_new_tokens new characters, one at a time."""
        # idx starts as (B, T) -- whatever context we're continuing from.
        for _ in range(max_new_tokens):
            logits, loss = self(idx)          # forward pass on everything generated so far
            logits = logits[:, -1, :]         # bigram model only uses the LAST character's
                                                # prediction; shape (B, T, C) -> (B, C)
            probs = F.softmax(logits, dim=-1) # convert raw scores to a probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # weighted random sample, (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # append the sampled character; loop continues
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")           # (B*T, vocab_size) after the internal flatten
print(f"Initial loss: {loss.item():.4f}")
# Sanity check: an untrained model's weights are random, so it's effectively guessing
# uniformly at random over the vocabulary. This computes the cross-entropy loss that
# PURE random guessing would produce -- if our actual initial loss doesn't roughly
# match this, something's wrong with initialization before we've spent any compute training.
print(f"Expected loss for random guessing: {-torch.log(torch.tensor(1.0/vocab_size)):.4f}")

If `Initial loss` is close to `Expected loss`, the model is correctly initialized (i.e. currently guessing uniformly at random over the vocabulary — it hasn't learned anything yet).

In [ ]:
# Generate from the UNTRAINED model -- expect pure gibberish, since the embedding
# table is still random weights at this point.
# Start from a single "newline" token (index 0) as the seed context: shape (1, 1) = (B=1, T=1).
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=100)
print(decode(generated[0].tolist()))  # generated[0] strips the batch dimension before decoding

### Train the Bigram model

In [ ]:
# AdamW: the standard optimizer for nearly all modern deep learning, including
# GPT-4-class models. Combines momentum (smooths past bumps in the loss landscape),
# per-parameter adaptive learning rates (rare characters get bigger updates than
# common ones), and weight decay (keeps weights small/stable, reduces overfitting).
# lr=1e-3 is a reasonably aggressive learning rate, fine for a model this tiny/simple.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32  # larger batch than our earlier demo (4) -- more stable gradient estimates

for steps in range(10000):
    xb, yb = get_batch('train')          # fresh random batch every step

    logits, loss = model(xb, yb)         # forward pass: predict + compute loss vs. ground truth

    # PyTorch accumulates gradients by default (adds new ones onto old ones), so we
    # must explicitly clear them before each backward pass -- otherwise gradients
    # from previous steps would corrupt this step's update.
    optimizer.zero_grad(set_to_none=True)

    loss.backward()                      # compute gradients: how would nudging each
                                           # weight up/down change the loss?
    optimizer.step()                     # apply those gradients to actually update the weights

    if steps % 1000 == 0:
        print(f"Step {steps}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")

In [ ]:
# Generate from the TRAINED model -- better than before, but still poor: the model
# has learned common CHARACTER-PAIR statistics (e.g. 't' is often followed by 'h'),
# but has no memory of anything beyond the single most recent character.
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))

**Checkpoint:** loss should drop from ~4.7 (random) to ~2.4-2.5. The output is *slightly* more English-shaped than noise but still gibberish — because the bigram model only ever looks at the single most recent character. It can't tell whether `h` came from `t` (predict `e`), `s` (predict `i`), or something else. That's exactly the problem self-attention solves next: letting characters look back at everything before them, not just the last one.

**Stop here and run this section before continuing** — confirm your loss curve and generated text look like the above, then we move to Part 9: the mathematical trick behind self-attention.